# Лабораторная работа №4. Случайный лес

Фокин Леонид Александрович, группа 401, вариант 20.

## Данные и метрики

Используются Wholesale Customers (`Channel`, macro-F1 и accuracy) и SGEMM (время ядра в мс, MAE и RMSE). Тестовая доля 20%, пять фолдов для подбора, `random_state=20`. Подробнее о наборах: [data/README.md](data/README.md). Основные метрики подбора: macro-F1 вверх, MAE вниз.

Загружаю общие функции разбиения и оценки. В этой работе библиотечный бейзлайн использует исходные числовые признаки.

In [1]:
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
import numpy as np
import pandas as pd
from IPython.display import display
from study import SEED, TASKS, evaluate, split, validate

BASE = 'raw'

def sklearn_model(task, params):
    if task == 'classification':
        return RandomForestClassifier(random_state=SEED, **params)
    return RandomForestRegressor(random_state=SEED, **params)

## Бейзлайн sklearn

Фиксирую качество моделей со стандартными параметрами. Тестовые оценки не участвуют в выборе конфигурации.

In [2]:
baseline = {task: evaluate(task, sklearn_model(task, {}), BASE) for task in TASKS}
display(pd.DataFrame([metrics for metrics, _ in baseline.values()]).round(4))

,task,f1_macro,accuracy,fit_seconds,mae_ms,rmse_ms
0,classification,0.9108,0.9205,0.0877,NaN,NaN
1,regression,NaN,NaN,0.4207,24.6465,48.6502


## Гипотезы и кросс-валидация

Число деревьев влияет на стабильность среднего прогноза. Меньшее число признаков на узле может снизить корреляцию деревьев, а больший лист сглаживает их прогнозы. Дополнительно проверяю, полезны ли признаки долей и плиток уже после усреднения деревьев.

В обоих случаях 5 фолдов. Сравниваю препроцессинг без подбора, подбор без изменения признаков и их комбинацию. Сначала выбираю лучшую альтернативу бейзлайну на CV, затем оцениваю её на отложенной части. `cv_std` показывает разброс оценок по фолдам, это не доверительный интервал.

In [3]:
feature_options = ['features']
class_grid = [{'n_estimators': 60, 'max_features': 'sqrt', 'min_samples_leaf': 1}, {'n_estimators': 100, 'max_features': 'sqrt', 'min_samples_leaf': 2}, {'n_estimators': 100, 'max_features': 'sqrt', 'min_samples_leaf': 4}, {'n_estimators': 60, 'max_features': 0.7, 'min_samples_leaf': 2}, {'n_estimators': 100, 'max_features': 0.7, 'min_samples_leaf': 4}, {'n_estimators': 150, 'max_features': 0.7, 'min_samples_leaf': 2}]
reg_grid = [{'n_estimators': 60, 'max_features': 'sqrt', 'min_samples_leaf': 1}, {'n_estimators': 100, 'max_features': 'sqrt', 'min_samples_leaf': 2}, {'n_estimators': 100, 'max_features': 'sqrt', 'min_samples_leaf': 4}, {'n_estimators': 60, 'max_features': 0.7, 'min_samples_leaf': 2}, {'n_estimators': 100, 'max_features': 0.7, 'min_samples_leaf': 4}, {'n_estimators': 150, 'max_features': 0.7, 'min_samples_leaf': 2}]
tables, chosen = {}, {}
for task, grid in zip(TASKS, (class_grid, reg_grid)):
    candidates = [('baseline', BASE, {})]
    candidates += [('preprocessing', prep, {}) for prep in feature_options]
    candidates += [('parameters', BASE, params) for params in grid]
    candidates += [('combined', prep, params) for prep in feature_options for params in grid]
    table = tables[task] = validate(task, lambda params: sklearn_model(task, params), candidates)
    chosen[task] = table.loc[table.experiment != 'baseline'].iloc[0]
    print(task)
    display(table.drop_duplicates('experiment').round(4).reset_index(drop=True))

classification


,experiment,prep,params,cv_mean,cv_std
0,combined,features,"{'n_estimators': 100, 'max_features': 0.7, 'mi...",0.9031,0.0161
1,preprocessing,features,{},0.8994,0.0295
2,parameters,raw,"{'n_estimators': 60, 'max_features': 0.7, 'min...",0.8963,0.0198
3,baseline,raw,{},0.8863,0.0275


regression


,experiment,prep,params,cv_mean,cv_std
0,preprocessing,features,{},25.4031,2.6554
1,combined,features,"{'n_estimators': 150, 'max_features': 0.7, 'mi...",27.4352,3.9293
2,baseline,raw,{},30.9162,2.7189
3,parameters,raw,"{'n_estimators': 150, 'max_features': 0.7, 'mi...",40.7412,5.5174


Лучший вариант из альтернатив заново обучаю на всём train. Прирост на CV не гарантирует прироста на тесте.

In [4]:
improved = {task: evaluate(task, sklearn_model(task, row.params), row.prep)
            for task, row in chosen.items()}
display(pd.DataFrame([metrics for metrics, _ in improved.values()]).round(4))

,task,f1_macro,accuracy,fit_seconds,mae_ms,rmse_ms
0,classification,0.9108,0.9205,0.1217,NaN,NaN
1,regression,NaN,NaN,0.5846,20.3524,42.4737


## Самописная реализация

Для каждого дерева беру бутстреп строк, на каждом узле выбираю случайное подмножество признаков. Регрессор усредняет прогнозы; классификатор усредняет вероятности классов в листьях, как sklearn. Генераторы случайных чисел воспроизводимы, но бутстрепы и последовательность разбиений двух реализаций не обязаны совпадать.

Самописный алгоритм проверяется с обоими рецептами sklearn: исходным и выбранным по CV. Приёмы обработки признаков общие, чтобы сравнить реализацию модели, а не разные входные данные.

In [5]:
from models.forest import Forest

def own_model(task, params):
    return Forest(task, random_state=SEED, **params)

own_base = {task: evaluate(task, own_model(task, {}), BASE) for task in TASKS}
own_improved = {task: evaluate(task, own_model(task, row.params), row.prep)
                for task, row in chosen.items()}

Сохраняю четыре сравнимые строки на задачу. `Δ` ниже означает разницу с соответствующим библиотечным или исходным вариантом, а не процентное изменение. Для MAE отрицательная разница означает улучшение.

In [6]:
versions = {'sklearn baseline': baseline, 'sklearn improved': improved,
            'own baseline': own_base, 'own improved': own_improved}
scores = pd.DataFrame([dict(version=version, **metrics)
                       for version, outcomes in versions.items()
                       for metrics, _ in outcomes.values()])
from pathlib import Path

Path('results').mkdir(exist_ok=True)
scores.drop(columns='fit_seconds').to_csv('results/lab4.csv', index=False)
for task, metric in zip(TASKS, ('f1_macro', 'mae_ms')):
    subset = scores.query('task == @task').dropna(axis=1).set_index('version')
    display(subset.drop(columns='task').round(4))
    s = subset[metric]
    display(pd.Series({'sklearn improved − sklearn baseline': s.iloc[1] - s.iloc[0],
                       'own baseline − sklearn baseline': s.iloc[2] - s.iloc[0],
                       'own improved − sklearn improved': s.iloc[3] - s.iloc[1],
                       'own improved − own baseline': s.iloc[3] - s.iloc[2]},
                      name=f'Δ {metric}').round(4))

,f1_macro,accuracy,fit_seconds
version,,,
sklearn baseline,0.9108,0.9205,0.0877
sklearn improved,0.9108,0.9205,0.1217
own baseline,0.8971,0.9091,0.2447
own improved,0.9108,0.9205,0.4566


sklearn improved − sklearn baseline    0.0000
own baseline − sklearn baseline       -0.0136
own improved − sklearn improved        0.0000
own improved − own baseline            0.0136
Name: Δ f1_macro, dtype: float64

,fit_seconds,mae_ms,rmse_ms
version,,,
sklearn baseline,0.4207,24.6465,48.6502
sklearn improved,0.5846,20.3524,42.4737
own baseline,29.8423,24.9637,51.2324
own improved,44.7007,21.0726,44.9119


sklearn improved − sklearn baseline   -4.2942
own baseline − sklearn baseline        0.3172
own improved − sklearn improved        0.7203
own improved − own baseline           -3.8911
Name: Δ mae_ms, dtype: float64

Сравниваю ошибки обеих улучшенных моделей на тех же тестовых объектах. Матрица показывает, какие классы путаются. В регрессии сравниваю величину ошибки на быстрых и медленных конфигурациях; медианный порог беру из train.

In [7]:
from sklearn.metrics import confusion_matrix

_, _, _, test_channel = split('classification')
_, _, train_time, test_time = split('regression')
bands = np.where(test_time <= train_time.median(), 'быстрые', 'медленные')
for label, outcomes in [('sklearn', improved), ('самописная', own_improved)]:
    print(label)
    display(pd.DataFrame(confusion_matrix(test_channel, outcomes['classification'][1], labels=[1, 2]),
                         index=['факт Horeca', 'факт Retail'],
                         columns=['прогноз Horeca', 'прогноз Retail']))
    error = np.abs(np.asarray(test_time) - outcomes['regression'][1])
    display(pd.Series(error).groupby(bands).agg(['size', 'mean']).round(2))

sklearn


,прогноз Horeca,прогноз Retail
факт Horeca,55,5
факт Retail,2,26


,size,mean
быстрые,308,7.03
медленные,292,34.40


самописная


,прогноз Horeca,прогноз Retail
факт Horeca,55,5
факт Retail,2,26


,size,mean
быстрые,308,7.11
медленные,292,35.80


## Выводы

- **Классификация.** Лучший совместный рецепт повысил CV macro-F1 с 0,8863 до 0,9031. На тесте обе версии sklearn получили 0,9108: прироста по метрикам нет. Самописный лес вырос с 0,8971 до 0,9108. У каждой версии свой набор бутстрепов, поэтому разница на одном разбиении не измеряет эффект реализации в отрыве от случайности.
- **Регрессия.** Дополнительные признаки без изменения параметров дали CV MAE 25,40 мс вместо 30,92. Варианты из сетки с ограничением числа признаков или размера листа не дали такого результата на исходных признаках. На тесте MAE sklearn снизилась с 24,65 до 20,35 мс, а самописного леса с 24,96 до 21,07 мс. Разрыв между улучшенными реализациями меньше 1 мс.